# Batch 5C — Rev. 4 Frontier API Baseline

This notebook runs the registered 36-row frontier-model comparison for RQ2. It uses the exact frozen questions, ordered evidence windows, free-form prompt/parser, ASK policy, and offline verifier, while replacing the Qwen answer call with the stable Gemini API model `gemini-3.5-flash`.

A GPU is not needed: inference runs through the API. Use a CPU runtime to preserve Colab compute units.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Before running

Create a **fresh** key in [Google AI Studio](https://aistudio.google.com/apikey). In Colab's **Secrets** panel (key icon), add it as `GEMINI_API_KEY` and enable notebook access. Revoke any key previously pasted into chat or a screenshot. The notebook never prints or saves the key. `gemini-3.5-flash` currently has a free tier, subject to Google's quota and availability limits.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPOSITORY_URL = 'https://github.com/Wenjia1215/ComplianceGPT.git'
SOURCE_COMMIT = 'a1643015dd5a9840b42b38e5989f885b6ed65c63'
REPO_DIR = Path('/content/ComplianceGPT_batch5c')
OUTPUT_DIR = Path('/content/drive/MyDrive/rq2_frontier_baseline_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if (REPO_DIR / '.git').is_dir():
    dirty = subprocess.check_output(
        ['git', 'status', '--porcelain'], cwd=REPO_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError(
            f'{REPO_DIR} contains local changes. Start a fresh runtime or move that directory.'
        )
    subprocess.run(['git', 'fetch', 'origin', 'main'], cwd=REPO_DIR, check=True)
else:
    subprocess.run(['git', 'clone', '--filter=blob:none', REPOSITORY_URL, str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_COMMIT], cwd=REPO_DIR, check=True)
checked_out = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out == SOURCE_COMMIT, (checked_out, SOURCE_COMMIT)
print('Pinned Batch 5C source:', checked_out)
print('Persistent result directory:', OUTPUT_DIR)

In [ ]:
import json
from importlib.metadata import version

runtime_manifest = OUTPUT_DIR / 'manifests/api_runtime.json'
packages = ['numpy==2.0.2', 'PyYAML==6.0.3']
if runtime_manifest.is_file():
    prior_runtime = json.loads(runtime_manifest.read_text(encoding='utf-8'))
    prior_google_genai = str(prior_runtime.get('google_genai_python', '') or '').strip()
    if prior_google_genai:
        packages.append(f'google-genai=={prior_google_genai}')
        print('Resuming with recorded Google Gen AI SDK:', prior_google_genai)
    else:
        packages.append('google-genai>=2.25.0,<3')
        print('Ignoring a pre-Gemini runtime manifest; the runner will replace it only if no API activity exists.')
else:
    packages.append('google-genai>=2.25.0,<3')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
print('Google Gen AI SDK:', version('google-genai'))

In [ ]:
import os
from google.colab import userdata

try:
    api_key = str(userdata.get('GEMINI_API_KEY') or '').strip()
except Exception as exc:
    raise RuntimeError(
        "Add GEMINI_API_KEY in Colab Secrets (key icon), enable notebook access, then rerun this cell."
    ) from exc
if not api_key:
    raise RuntimeError('GEMINI_API_KEY is empty in Colab Secrets.')
os.environ['GEMINI_API_KEY'] = api_key
del api_key
print('GEMINI_API_KEY loaded into process memory; its value was not displayed or written.')

In [ ]:
# Discover repository files directly to avoid third-party packages named tests.
test_environment = os.environ.copy()
test_environment['PYTHONPATH'] = str(REPO_DIR / 'src')
test_environment['PYTHONDONTWRITEBYTECODE'] = '1'
test_directory = REPO_DIR / 'tests'
for test_name in ('test_frontier_api_answerer.py', 'test_rq2_frontier_baseline.py'):
    if not (test_directory / test_name).is_file():
        raise FileNotFoundError(f'Batch 5C test file is missing: {test_directory / test_name}')
test_command = [
    sys.executable, '-m', 'unittest', 'discover',
    '-s', str(test_directory), '-p', 'test_*frontier*.py', '-v',
]
test_result = subprocess.run(
    test_command, cwd=REPO_DIR, env=test_environment,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)
print(test_result.stdout, end='')
if test_result.returncode != 0:
    raise RuntimeError(
        f'Batch 5C offline tests failed (exit {test_result.returncode}). '
        'See the unittest output above for the underlying error.'
    )
print('Batch 5C offline tests passed.')


In [ ]:
import csv
import hashlib
from IPython.display import Markdown, display

FROZEN_ARCHIVE_SHA256 = '56a70db6eca0420df6affbe63c859418ac423b54d80d3d1eae7bf785f0f63328'
frozen_archive = REPO_DIR / 'experiments/answerer_comparison/rq2_matched/results_v3/compliancegpt_rq2_matched_v3.zip'
actual_archive_sha256 = hashlib.sha256(frozen_archive.read_bytes()).hexdigest()
assert actual_archive_sha256 == FROZEN_ARCHIVE_SHA256, (
    actual_archive_sha256, FROZEN_ARCHIVE_SHA256
)
runner = REPO_DIR / 'experiments/answerer_comparison/rq2_frontier_baseline/run_frontier_baseline.py'
prepare_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
    '--prepare-only',
]
subprocess.run(prepare_command, cwd=REPO_DIR, check=True)
contexts_path = OUTPUT_DIR / 'contexts/rev4_prepared_contexts.jsonl'
with contexts_path.open('r', encoding='utf-8') as handle:
    prepared_rows = sum(1 for line in handle if line.strip())
assert prepared_rows == 36, f'Expected 36 prepared contexts, found {prepared_rows}'
preflight_summary = json.loads((OUTPUT_DIR / 'preflight/summary.json').read_text(encoding='utf-8'))
assert preflight_summary.get('scope') == 'gold-free frozen-context integrity audit; no API call'
assert preflight_summary.get('api_calls_performed') == 0
print('Frozen archive verified:', actual_archive_sha256)
print(f'Prepared-context audit passed: {prepared_rows} Rev. 4 contexts; no API call made.')
display(Markdown((OUTPUT_DIR / 'PREPARED_CONTEXTS.md').read_text(encoding='utf-8')))

## Run the registered API comparison

The first-attempt prompts contain about 227,000 characters in total (roughly 57,000 tokens by a simple character estimate). Actual API token usage is recorded from each response. A parse failure can trigger up to two additional calls for that row.

The cell checkpoints every completed row and every returned API response. If Google returns a transient per-minute HTTP 429 with a retry delay, the cell waits at least 65 seconds and resumes automatically; validated result rows are skipped. Other failures still stop immediately. After a runtime disconnect, reconnect and use **Runtime → Run all**.

Google's current pricing table states that free-tier content may be used to improve its products. Gold labels and the API key are never included in prompt content.

In [ ]:
from collections import deque
import math
import re
import time

def per_minute_quota_wait_seconds(output_text):
    if '429 RESOURCE_EXHAUSTED' not in output_text:
        return None
    if 'PerMinute' not in output_text and 'requests per minute' not in output_text.lower():
        return None
    matches = []
    for pattern in (
        r'Please retry in\s+([0-9]+(?:\.[0-9]+)?)s',
    ):
        matches.extend(float(value) for value in re.findall(pattern, output_text))
    server_delay = max(matches, default=60.0)
    return max(65, math.ceil(server_delay + 5.0))

run_command = [
    sys.executable, '-u', str(runner),
    '--repo-root', str(REPO_DIR),
    '--output-dir', str(OUTPUT_DIR),
]
run_environment = os.environ.copy()
maximum_quota_restarts = 50
quota_restarts = 0
failure_log = OUTPUT_DIR / 'runner_failure_tail.log'
print('Starting the 36-row Rev. 4 Gemini 3.5 Flash API baseline.')
print('Transient per-minute 429 responses will wait and resume automatically.')
print('After a disconnect, reconnect and Run all; validated rows are skipped.')
while True:
    tail = deque(maxlen=400)
    process = subprocess.Popen(
        run_command, cwd=REPO_DIR, env=run_environment,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        tail.append(line)
    return_code = process.wait()
    if return_code == 0:
        break
    failure_output = ''.join(tail)
    failure_log.write_text(failure_output, encoding='utf-8')
    wait_seconds = per_minute_quota_wait_seconds(failure_output)
    if wait_seconds is None or quota_restarts >= maximum_quota_restarts:
        print(f'Batch 5C failed. The final runner output is saved at {failure_log}.')
        raise subprocess.CalledProcessError(return_code, run_command)
    quota_restarts += 1
    print(
        f'Per-minute Gemini quota reached; waiting {wait_seconds}s before resumable restart '
        f'{quota_restarts}/{maximum_quota_restarts}. Completed rows remain checkpointed.'
    )
    time.sleep(wait_seconds)
if failure_log.exists():
    failure_log.unlink()
print('Batch 5C runner completed successfully.')

In [ ]:
summary_path = OUTPUT_DIR / 'summary.json'
api_manifest_path = OUTPUT_DIR / 'manifests/api_responses.json'
assert summary_path.is_file(), summary_path
assert api_manifest_path.is_file(), api_manifest_path
summary = json.loads(summary_path.read_text(encoding='utf-8'))
api_manifest = json.loads(api_manifest_path.read_text(encoding='utf-8'))
assert summary.get('n_questions') == 36
assert api_manifest.get('completed_result_rows') == 36
display(Markdown((OUTPUT_DIR / 'SUMMARY.md').read_text(encoding='utf-8')))
print('Server-reported model:', api_manifest['response_model'])
print('All logged API usage:', api_manifest['all_logged_usage']['totals'])
print('Parse-retry rows:', api_manifest['parse_retry_rows'])
print('Orphaned API calls:', api_manifest['orphaned_call_count'])

In [ ]:
archive_path = OUTPUT_DIR.parent / f'{OUTPUT_DIR.name}.zip'
assert archive_path.is_file(), f'Result ZIP was not created: {archive_path}'
archive_sha256 = hashlib.sha256(archive_path.read_bytes()).hexdigest()
print('Confirmed result directory:', OUTPUT_DIR)
print('Confirmed result ZIP:', archive_path)
print('Result ZIP SHA-256:', archive_sha256)
print('Download this executed notebook from Colab and attach it together with the ZIP for independent audit.')

## Handoff

After the final cell succeeds, download this executed `.ipynb` from Colab and provide it together with `/content/drive/MyDrive/rq2_frontier_baseline_v1.zip`. The executed notebook, raw API log, row-level contracts, summary, and hashes are all needed before the result is added to the repository.